# 🧱 Part 3：Token ID → 向量 — Embedding + Position Encoding

> **前情回顾**：Tokenizer 把文本变成了数字序列，比如 `"the cat" → [3, 7, 1, 12, 5]`。
>
> **现在的问题**：这些数字怎么喂给神经网络？3 比 7 小，所以 "the" 比 "cat" 不重要？显然不对。
>
> 本 Part 目标：**理解 token ID 如何变成神经网络能「理解」的向量，以及如何让模型知道 token 之间的顺序。**

In [ ]:
import torch
import torch.nn as nn
import math
import matplotlib.pyplot as plt

torch.manual_seed(42)
print(f"PyTorch 版本: {torch.__version__}")

### 0. 先建立直觉：为什么词能变成空间中的点？

在进入代码之前，先想清楚一个根本问题：**一个词，凭什么能被压缩成几个数字？这几个数字凭什么能「代表」这个词的意思？**

---

#### 从「描述一个人」讲起

假如只能用**一个数字**描述你，你选什么？身高？体重？年龄？

无论选哪个，都会丢失大量信息。168cm 的人有很多，他们完全不一样。

但如果用**多个数字**描述呢？

```
你 = [身高, 体重, 年龄, 收入, 性格外向度, 学历, 所在城市规模, ...]
     [168,   65,   28,   20k,  0.7,      硕士,  300万,         ...]
```

维度越多，描述越精确。**在高维空间里，每个人就是唯一的一个点。**

词也一样。

---

#### 一个词的「人格」

一个词的意思，不是字典里的一行定义，而是**它在无数句子中和别的词怎么搭配**。

```
"cat" 的行为：
  - 经常和 "sat", "pet", "meow", "fur" 一起出现
  - 很少和 "algorithm", "stock", "galaxy" 一起出现
  - 可以做主语、宾语
  - 前面常有 "the", "a", "my"

"dog" 的行为：
  - 经常和 "sat", "pet", "bark", "fur" 一起出现
  - 很少和 "algorithm", "stock", "galaxy" 一起出现
  - 可以做主语、宾语
  - 前面常有 "the", "a", "my"
```

发现了没？**"cat" 和 "dog" 的行为模式几乎一模一样。** 它们出现在类似的上下文里。

这就是语言学中的**分布式假设**：**你只要告诉我一个词在什么上下文里出现，我就知道它是什么意思。**

---

#### 高维空间里发生了什么

想象一个很高很高的空间（比如 512 维），里面的每个方向代表某种「语言特征」：

```
维度 0： 动物性 ←────────────→ 物体性
维度 1： 动作性 ←────────────→ 状态性
维度 2： 积极   ←────────────→ 消极
维度 3： 正式   ←────────────→ 口语
...
维度 511: 某个无法命名的细微语义
```

**没有人手动设计这些维度**——它们是训练时自己涌现出来的。

结果是：
- "cat" 和 "dog" 在「动物性」维度上位置相近 → 两个点在空间里挨着
- "cat" 和 "algorithm" 在每个维度上都不像 → 两个点相距很远
- **"king" - "man" + "woman" ≈ "queen"** ——方向 "man→woman" 编码了性别变换

```
高维空间中的关系示意（降维到 3D 想象）：

        🐕 dog
       / 
   🐈 cat    ← 动物区
  /
 ────────────────────  💻 algorithm  ← 另一端
```

---

#### 那 Embedding 在做什么？

Embedding 就是给每个词分配一个**高维空间中的初始坐标**（随机给的）。

训练过程中：
- 每当 "cat" 和 "dog" 出现在相似的上下文 → 它们的坐标被拉近
- 每当 "cat" 和 "algorithm" 出现在完全不同的上下文 → 它们的坐标被推远

几百万次调整之后，**这个空间就有了「语义结构」**——相似词的坐标天然靠近。

这就是 Embedding 的本质：**把离散的符号（词 ID）映射到连续的语义空间（向量）**。

---

接下来我们看代码怎么实现这个映射。

### 1. 第一个问题：token ID 为什么不能直接用？

Tokenizer 给了我一个数字列表：`[3, 7, 1, 12, 5]`。能不能直接把这串数字喂给神经网络？

**不能。** 因为这些数字只是「编号」，3 不代表比 7 更不重要，7 也不代表是 3 的两倍多。

```
ID=3 → 代表 "the"
ID=7 → 代表 "cat"
ID=12 → 代表 "mat"
```

如果我们直接用数字大小，模型会误以为 `"mat"(12) > "cat"(7) > "the"(3)`。
这完全没道理。

**我们需要把每个 token ID 变成一个「高维空间中的坐标」。** 在这个高维空间里：
- "cat" 和 "dog" 的坐标可能很近（都是动物）
- "cat" 和 "mat" 的坐标可能有一定关系（经常一起出现）
- "cat" 和 "log" 就没什么关系

这个转换就叫 **Embedding**。

### 2. Embedding 的本质：一张巨大的查表

`nn.Embedding` 听起来很高端，但它的本质就是：

```
一个矩阵：[vocab_size × embed_dim]

   embed_dim (比如 8 维)
   ↓   ↓   ↓   ↓   ↓   ↓   ↓   ↓
  [0.1, 0.3,-0.2, 0.5, 0.8,-0.1, 0.2,-0.4]  ← token 0 的向量
  [0.4,-0.1, 0.7,-0.3, 0.2, 0.6,-0.5, 0.1]  ← token 1 的向量
  [-0.2,0.5, 0.1, 0.9,-0.3, 0.4,-0.1, 0.7]  ← token 2 的向量
  ...
  [0.6,-0.3, 0.2, 0.1,-0.7, 0.5, 0.3,-0.2]  ← token N 的向量
```

- **查表操作**：给一个 token ID，取矩阵的第 ID 行，就是它的向量
- `embedding[3]` = 矩阵的第 3 行
- 就这么简单！

这些数字一开始是**随机初始化的**，然后通过训练反向传播不断调整。训练完成后，「意思相近的词」会被拉到空间中接近的位置。

In [ ]:
# 模拟一个 mini 词表，Embedding = vocab_size × embed_dim 的矩阵
vocab = ["the", "cat", "sat", "on", "mat", "dog", "log"]
vocab_size = len(vocab)
embed_dim = 4

embedding = nn.Embedding(vocab_size, embed_dim)

print(f"词表大小: {vocab_size}, Embedding 维度: {embed_dim}")
print(f"Embedding 权重形状: {embedding.weight.shape}  ← 就是一个 {vocab_size}×{embed_dim} 矩阵")
print(f"\n前 3 行初始值（随机）:\n{embedding.weight[:3]}")

In [ ]:
# 查表：给一组 token ID，取出对应的向量
sentence_ids = torch.tensor([0, 1, 2, 3, 0, 4])  # "the cat sat on the mat"
vectors = embedding(sentence_ids)                  # 查表 → [6, 4]

print(f"token IDs: {sentence_ids.tolist()}  →  {[vocab[i] for i in sentence_ids.tolist()]}")
print(f"输出形状: {vectors.shape}  ← [{len(sentence_ids)} 个 token, 每个 {embed_dim} 维]")
print()

# 逐个看
for i, (tid, vec) in enumerate(zip(sentence_ids.tolist(), vectors)):
    print(f"  位置 {i}: '{vocab[tid]}' (ID={tid}) → {vec.tolist()}")

# 关键观察：位置 0 和 4 都是 'the'，向量一模一样
# → 模型无法区分两个位置的同一个词 → 需要 Position Encoding
print(f"\n注意：位置 0 和 4 的向量完全相同 ← 这引出了 Position Encoding 的必要性")

### 3. 第二个问题：模型怎么知道 token 的顺序？

仔细看上一步的输出：
- 位置 0 的 `"the"` → 向量 `[0.xx, ...]`
- 位置 4 的 `"the"` → 向量 `[0.xx, ...]`（一模一样！）

**模型眼里，这两个 "the" 是同一个人。**

但实际上：
- `"the cat"` 和 `"cat the"` 意思完全不同
- `"I love you"` 和 `"You love I"` 是两回事

Embethding 只管「你是谁」，不管「你在哪」。

**解决方案：给每个位置也分配一个向量，叫 Position Encoding。**

```
最终输入 = Token Embedding + Position Encoding

"the" 在位置 0 的最终表示 = Embedding['the'] + Position[0]
"the" 在位置 4 的最终表示 = Embedding['the'] + Position[4]
                                   ↑ 这里不同！
```

### 4. 正弦位置编码（Sinusoidal Position Encoding）

Transformer 论文用的位置编码是**手工设计的正弦/余弦函数**，不是学出来的。

公式：
```
PE(pos, 2i)   = sin(pos / 10000^(2i/d))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d))

pos = 位置编号 (0, 1, 2, ...)
i   = 维度编号 (0, 1, 2, ..., d/2 - 1)
d   = embedding 维度
```

看起来很 scary，但本质是：**每个位置有一组不同频率的正弦/余弦波，把这些波的取值拼起来就是这个位置的编码。**

我们先实现再看。

In [ ]:
def get_sinusoidal_encoding(seq_len, d_model):
    """
    正弦位置编码：每个位置用不同频率的 sin/cos 波生成唯一向量
    
    PE(pos, 2i)   = sin(pos / 10000^(2i/d))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d))
    """
    position = torch.arange(seq_len).unsqueeze(1)  # [seq_len, 1]
    div_term = torch.exp(
        torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
    )
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)   # 偶数维 sin
    pe[:, 1::2] = torch.cos(position * div_term)   # 奇数维 cos
    return pe

# 生成 10 个位置、8 维的编码
pe = get_sinusoidal_encoding(seq_len=10, d_model=8)
print(f"位置编码形状: {pe.shape}  ← 10 个位置 × 每个位置 8 维")
print(f"前 3 个位置的编码:\n{pe[:3]}")

### 5. 可视化位置编码

文字看不懂，直接看图。

In [ ]:
# 可视化位置编码：热力图 + 波形图
pe_viz = get_sinusoidal_encoding(seq_len=50, d_model=32)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左: 热力图 — 横轴=维度, 纵轴=位置, 色=值
im = axes[0].imshow(pe_viz.numpy(), aspect='auto', cmap='RdBu')
axes[0].set_xlabel('Embedding 维度 (0-31)'); axes[0].set_ylabel('位置 (0-49)')
axes[0].set_title('位置编码热力图\n(蓝=-1, 红=+1)')
plt.colorbar(im, ax=axes[0])

# 右: 选几个维度，看它们的值随位置怎么变
# 低维度波长短（变化快→区分邻居），高维度波长长（变化慢→区分远处）
for dim_idx in [0, 1, 4, 8, 16]:
    axes[1].plot(range(50), pe_viz[:, dim_idx].numpy(), label=f'维度 {dim_idx}')
axes[1].set_xlabel('位置'); axes[1].set_ylabel('编码值')
axes[1].set_title('不同维度随位置变化的波形')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 6. 完整的 Embedding + Position Encoding 模块

把 Token Embedding 和 Position Encoding 整合成一个 PyTorch Module。

In [ ]:
class TokenEmbedding(nn.Module):
    """Token ID → 向量：Embedding 查表 + 位置编码"""
    
    def __init__(self, vocab_size, d_model, max_seq_len=512):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        # Position Encoding 手工算，不参与训练（register_buffer）
        pe = get_sinusoidal_encoding(max_seq_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        # x: [batch, seq_len] → output: [batch, seq_len, d_model]
        seq_len = x.shape[1]
        token_vecs = self.token_emb(x)         # 查表
        pos_vecs = self.pe[:seq_len, :]        # 取对应位置编码
        return token_vecs + pos_vecs            # 加法（不是拼接）

# 测试：batch=2, 每条 5 个 token
test_vocab_size, test_d_model = 20, 8
emb_module = TokenEmbedding(test_vocab_size, test_d_model)

dummy_input = torch.randint(0, test_vocab_size, (2, 5))
output = emb_module(dummy_input)

print(f"输入形状: {dummy_input.shape}  →  输出形状: {output.shape}")
print(f"→ [batch=2, seq_len=5, d_model={test_d_model}]")

# 关键验证：同一个 token 在不同位置，向量不同（因为加了不同位置编码）
print(f"\n验证：token ID=0 在不同位置的向量前 3 维")
print(f"  位置 0: {output[0, 0, :3].tolist()}")
print(f"  位置 3: {output[0, 3, :3].tolist()}")
print(f"  → 不同！同一个词在不同位置有了不同的表示")

### 7. 一个小 trick：为什么是「加」而不是「拼」？

你可能会想：「为什么要把 Token Embedding 和 Position Encoding **加**起来？为什么不拼接（concatenate）？」

两个原因：
1. **维度效率**：拼接会让维度翻倍 → 后面所有层的计算量都翻倍 → 不划算
2. **加法就够了**：理论上，两个向量相加后，模型可以通过后续的线性变换分离它们。实际操作中，加法效果和拼接差不多

所以业界都用加法。

### 8. batch 的概念：为什么我们需要 batch 维度？

你可能注意到我们的输入从 `[seq_len]` 变成了 `[batch_size, seq_len]`。

```
batch_size = 2 → 同时处理 2 个句子

样本 0: [the, cat, sat, on, the]  →  ID: [0, 1, 2, 3, 0]
样本 1: [i,   love, my,  dog]    →  ID: [5, 6, 7, 8]

拼成一个 batch:
[[0, 1, 2, 3, 0],
 [5, 6, 7, 8, 0]]  ← 短的补 0（padding）

形状: [2, 5] = [batch_size, seq_len]
```

为什么要 batch？**GPU 喜欢同时算很多条，效率更高。** 就像食堂打饭，一次打 10 个人的饭比打 10 次单独一个人的饭快得多。

In [ ]:
# 直观感受 batch：一次处理多条样本
batch = torch.tensor([
    [0, 1, 2, 3, 0],  # the cat sat on the
    [1, 5, 4, 0, 0],  # cat dog mat the the (padding)
])

emb_output = emb_module(batch)
print(f"输入形状: {batch.shape} = [batch_size=2, seq_len=5]")
print(f"输出形状: {emb_output.shape} = [2, 5, d_model={test_d_model}]")
print(f"→ GPU 同时算 2 条，比一条一条算快")

### 9. 可视化：未训练 vs 已训练的 Embedding

上面说了那么多「训练后语义相近的词会聚在一起」，到底长什么样？

下面用 t-SNE 把 16 维的 embedding 压到 2 维，直接看图对比。

In [ ]:
# === 构建词表：14 个词，分 5 个语义组 + 1 个特殊 token ===
words = [
    "cat", "dog", "bird",          # 动物
    "sat", "ran", "flew",          # 动作
    "on", "under",                 # 介词
    "mat", "rug", "tree", "sky",   # 物品
    "the", "a",                    # 冠词
    "[PAD]",                       # 特殊 token
]
vocab_size = len(words)
d_model = 16

word_groups = {
    "cat": "动物", "dog": "动物", "bird": "动物",
    "sat": "动作", "ran": "动作", "flew": "动作",
    "on": "介词", "under": "介词",
    "mat": "物品", "rug": "物品", "tree": "物品", "sky": "物品",
    "the": "冠词", "a": "冠词",
    "[PAD]": "特殊",
}

# === 未训练的 Embedding（随机初始化）===
# 词的位置完全随机，没有任何语义结构
torch.manual_seed(42)
untrained_emb = nn.Embedding(vocab_size, d_model)

# === 模拟「已训练」的 Embedding ===
# 同组词共享基础向量 + 小噪声 → 同组词在空间中自然聚在一起
# [PAD] 放在远离所有词的位置（训练中它不和任何词共现，自然被推开）
torch.manual_seed(999)
group_bases = {g: torch.randn(d_model) * 0.6
               for g in ["动物", "动作", "介词", "物品", "冠词"]}

trained_emb = nn.Embedding(vocab_size, d_model)
with torch.no_grad():
    for i, w in enumerate(words):
        group = word_groups[w]
        if group == "特殊":
            trained_emb.weight[i] = torch.tensor([10.0, 10.0] + [0.0] * (d_model - 2))
        else:
            trained_emb.weight[i] = group_bases[group] + 0.12 * torch.randn(d_model)

# === t-SNE 降维到 2D ===
# 两组 embedding 拼在一起降维 → 同一坐标系，公平对比
all_emb = torch.cat([untrained_emb.weight.data, trained_emb.weight.data], dim=0)

try:
    from sklearn.manifold import TSNE
    tsne = TSNE(n_components=2, random_state=42, perplexity=5)
    all_2d = tsne.fit_transform(all_emb.numpy())
except ImportError:
    from sklearn.decomposition import PCA
    pca = PCA(n_components=2)
    all_2d = pca.fit_transform(all_emb.numpy())

n = vocab_size
untrained_2d = all_2d[:n]   # 前 n 行 = 未训练
trained_2d = all_2d[n:]     # 后 n 行 = 已训练

# === 画图：左右对比 ===
group_colors = {
    "动物": "#e74c3c", "动作": "#2ecc71", "介词": "#3498db",
    "物品": "#f39c12", "冠词": "#9b59b6", "特殊": "#95a5a6",
}
colors = [group_colors[word_groups[w]] for w in words]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 左图：未训练 — 词随机散落，同类词不一定靠近
axes[0].scatter(untrained_2d[:, 0], untrained_2d[:, 1], c=colors, s=200,
                edgecolors='black', linewidth=1.2, alpha=0.85, zorder=5)
for i, w in enumerate(words):
    axes[0].annotate(w, (untrained_2d[i, 0], untrained_2d[i, 1]),
                     fontsize=10, textcoords="offset points", xytext=(6, 8 if w != "[PAD]" else -15))
axes[0].set_title("未训练的 Embedding（t-SNE）\n随机初始化，词的位置毫无规律", fontsize=14, fontweight='bold')
axes[0].set_xlabel("t-SNE 维度 1"); axes[0].set_ylabel("t-SNE 维度 2")
axes[0].grid(True, alpha=0.15)

# 右图：已训练 — 同颜色 = 同语义组，贴在一起
axes[1].scatter(trained_2d[:, 0], trained_2d[:, 1], c=colors, s=200,
                edgecolors='black', linewidth=1.2, alpha=0.85, zorder=5)
for i, w in enumerate(words):
    axes[1].annotate(w, (trained_2d[i, 0], trained_2d[i, 1]),
                     fontsize=10, textcoords="offset points", xytext=(6, 8 if w != "[PAD]" else -15))
axes[1].set_title("训练后的 Embedding（t-SNE）\n同类词挤在一起，不同类分离", fontsize=14, fontweight='bold')
axes[1].set_xlabel("t-SNE 维度 1"); axes[1].set_ylabel("t-SNE 维度 2")
axes[1].grid(True, alpha=0.15)

from matplotlib.patches import Patch
axes[1].legend(
    handles=[Patch(facecolor=c, edgecolor='black', label=g) for g, c in group_colors.items()],
    loc='lower right', fontsize=10, title="语义分组", title_fontsize=11)

plt.tight_layout()
plt.show()

# 肉眼可见的对比：
#  左：cat/dog/bird 散落在不同角落，没有任何结构
#  右：cat/dog/bird（红）挤成一团，sat/ran/flew（绿）聚在另一块，[PAD] 被推远

---

## 第三部分小结

现在你对「token ID → 向量」这条链路应该完全通了：

1. ✅ Token ID 只是个编号，不能直接用——没有大小意义
2. ✅ `nn.Embedding` = 一张矩阵，用 token ID 做行号查表取向量
3. ✅ Embedding 管「你是谁」，但不管「你在哪」
4. ✅ Position Encoding 给每个位置一个独特的「位置指纹」
5. ✅ 最终输入 = Token Embedding + Position Encoding（加法，不拼接）
6. ✅ batch 维度：一次处理多条样本，GPU 并行加速

**你可能会想：这些向量如果两个相似的词向量很接近，那模型就「理解」了词的关系吗？**

答案是：现在还不够。目前 embedding 里的向量都是随机初始化的，它们变成有意义的向量是**在训练过程中**逐渐发生的。而训练怎么做的？Self-Attention 怎么做的？

→ 下一 Part：手写 Transformer 的核心 —— Self-Attention！